In [ ]:
#This code is used if the notebook is implemented in github codespace. Just remove the (#)
!python -m pip install .. --quiet

# Step 1: Import GEE account setting from luma-stack

In [ ]:
# Use EE initialization from luma_ge
import ee 
import luma_ge

# Autheticate using service account (json file)

service_account_path = '../auth/ee-epstm2024.json'
success = luma_ge.initialize_with_service_account(service_account_path)

if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")


# Step 2: Upload AOI and collect satellite images

In [ ]:
# Collect satellite images

import geemap
from luma_ge.input_utils import shapefile_validator, kml_validator
from luma_ge.data_acquisition import Reflectance_Data, Reflectance_Stats, final_Image
import geopandas as gpd

In [ ]:
# aoi = geemap.shp_to_ee("../data/test_data/area_of_interest.shp")

# #initilizae the validator
validator_shp = shapefile_validator(verbose=True)
#Path to test shapefile. can be change accordingly
shapefile_path = '../data/test_data/area_of_interest.shp'
#for demo, used geopandas to load the shapefile
test_gdf = gpd.read_file(shapefile_path)
print(f"\nLoaded shapefile with {len(test_gdf)} features")
print(f"Geometry types: {test_gdf.geometry.geom_type.unique()}")

#Validate and fix geometry
print("\n--- Starting Validation ---")
validated_gdf = validator_shp.validate_and_fix_geometry(test_gdf, geometry="mixed")

if validated_gdf is not None:
    print(f"\n✓ Validation successful!")
    print(f"Features after validation: {len(validated_gdf)}")
    print(f"All geometries valid: {validated_gdf.geometry.is_valid.all()}")
else:
    print(f"\n✗ Validation failed!")

aoi = geemap.shp_to_ee("../data/test_data/area_of_interest.shp")

In [ ]:
#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2024-01-01'
end = '2024-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True) #REPLACE OLD CODE WITH THE NEW ONE HERE
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
#Add the data to the map
Map = geemap.Map()
Map.addLayer(mosaic_landsat, l8_sr_visparam, 'L8 SR Mosaic')
Map.addLayer(median_landsat, l8_sr_visparam, 'L8 SR Median')
Map.addLayer(landsat_data, l8_sr_visparam, 'L8 SR Image Collection')
# set center of the map in the area of interest
Map.centerObject(aoi, 7)

#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
thermal_vis = {'min': 286,'max': 300,'gammma': 0.4}
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()
#visualize the thermal bands and multispectral bands
Map.addLayer(median_thermal, thermal_vis, "Thermal Bands")
#Map

# Step 3: Upload modular reference data

This step uploads the reference data that contains UML information inside its attribute table

## Create dummy modular reference data

In [ ]:
# import ee


# class SyntheticModularTrainingData:
#     """
#     Create synthetic modular training dataset for testing primitive layers.
#     Assumes Earth Engine already initialized.
#     """

#     def __init__(self, aoi, n_points=200, seed=42):

#         if not isinstance(aoi, ee.Geometry):
#             raise ValueError("AOI must be ee.Geometry")

#         self.aoi = aoi
#         self.n_points = n_points
#         self.seed = seed

#     # -------------------------------------------------
#     # Generate random points
#     # -------------------------------------------------

#     def _generate_points(self):

#         pts = ee.FeatureCollection.randomPoints(
#             region=self.aoi,
#             points=self.n_points,
#             seed=self.seed
#         )

#         return pts

#     # -------------------------------------------------
#     # Add random columns
#     # -------------------------------------------------

#     def _add_random_columns(self, fc):

#         fc = fc.randomColumn("r1", self.seed)
#         fc = fc.randomColumn("r2", self.seed + 1)
#         fc = fc.randomColumn("r3", self.seed + 2)
#         fc = fc.randomColumn("r4", self.seed + 3)
#         fc = fc.randomColumn("r5", self.seed + 4)

#         return fc

#     # -------------------------------------------------
#     # Assign labels
#     # -------------------------------------------------

#     def _assign_labels(self, feature):

#         tree = ee.Number(feature.get("r1")).gt(0.5)
#         built = ee.Number(feature.get("r2")).gt(0.7)
#         soil = ee.Number(feature.get("r3")).gt(0.6)
#         water = ee.Number(feature.get("r4")).gt(0.8)

#         tree_temp = ee.Algorithms.If(
#             tree,
#             ee.Number(feature.get("r5")).gt(0.5),
#             0
#         )

#         return feature.set({
#             "tree_presence": tree,
#             "builtupsurface_presence": built,
#             "baresoil_presence": soil,
#             "waterbody_presence": water,
#             "tree_temporal_variation": tree_temp
#         })

#     # -------------------------------------------------
#     # Public
#     # -------------------------------------------------

#     def create(self):

#         pts = self._generate_points()

#         pts = self._add_random_columns(pts)

#         labeled = pts.map(self._assign_labels)

#         labeled = labeled.select([
#             "tree_presence",
#             "builtupsurface_presence",
#             "baresoil_presence",
#             "waterbody_presence",
#             "tree_temporal_variation"
#         ])

#         return labeled

In [ ]:
# synthetic = SyntheticModularTrainingData(
#     aoi=aoi,
#     n_points=300
# )

# training_fc = synthetic.create()

# preview = training_fc.limit(10).getInfo()

# for f in preview["features"]:
#     print(f["properties"])

# inside_test = training_fc.map(
#     lambda f: f.set(
#         "inside",
#         ee.Geometry(aoi).contains(f.geometry())
#     )
# )

# print(
#     inside_test.aggregate_histogram("inside").getInfo()
# )

## Upload modular reference data

In [ ]:
import geopandas as gpd
import ee
from shapely.geometry import shape


def load_modular_training_data(
    shp_path,
    aoi=None,
):
    """
    Load shapefile with LCML attributes
    and filter by AOI (EE FeatureCollection).
    """

    # -------------------------
    # read shapefile
    # -------------------------

    gdf = gpd.read_file(shp_path)

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    gdf = gdf.to_crs("EPSG:4326")

    # -------------------------
    # AOI filter (EE FeatureCollection)
    # -------------------------

    if aoi is not None:

        # convert EE FeatureCollection → geometry → shapely
        aoi_geom = aoi.geometry().getInfo()

        aoi_shape = shape(aoi_geom)

        gdf = gdf[gdf.intersects(aoi_shape)]

    # -------------------------
    # convert to EE FeatureCollection
    # -------------------------

    features = []

    for _, row in gdf.iterrows():

        geom = ee.Geometry(row.geometry.__geo_interface__)

        props = row.drop("geometry").to_dict()

        features.append(
            ee.Feature(geom, props)
        )

    ee_fc = ee.FeatureCollection(features)

    # -------------------------
    # output
    # -------------------------

    return {
        "gdf": gdf,
        "ee_fc": ee_fc,
        "columns": list(gdf.columns),
        "size": len(gdf),
    }

In [ ]:
data = load_modular_training_data(
    shp_path="../data/test_data/training_points.shp",
    aoi=aoi
)

training_gdf = data["gdf"]

training_fc = data["ee_fc"]

print(data["columns"])
print(data["size"])

# Step 4: Generate element layers

In [ ]:
from luma_ge.classification import FeatureExtraction
from luma_ge.classification import Generate_LULC

class PrimitiveLayerTrainer:

    def __init__(self, image, roi):

        self.image = image
        self.roi = roi

        self.primitives = self._get_primitives_from_training()


    # ---------------------------------
    # detect primitives from attributes
    # ---------------------------------

    def _get_primitives_from_training(self):

        props = (
            self.roi.first()
            .propertyNames()
            .getInfo()
        )

        exclude = ["LULC_Type", "ID", "geometry", "system:index"]

        primitives = [
            p for p in props
            if p not in exclude
        ]

        print("Detected primitives:", primitives)

        return primitives


    # ---------------------------------
    # train one primitive
    # ---------------------------------

    def train_one(self, primitive):

        # remove system:index
        roi_clean = self.roi.map(
            lambda f: f.select(
                f.propertyNames().remove("system:index")
            )
        )

        sample = self.image.sampleRegions(
            collection=roi_clean,
            properties=[primitive],
            scale=30,
            geometries=False
        )

        classifier = ee.Classifier.smileRandomForest(50)

        trained = classifier.train(
            features=sample,
            classProperty=primitive,
            inputProperties=self.image.bandNames()
        )

        result = self.image.classify(trained)

        return result.rename(primitive)

    # ---------------------------------
    # train all primitives
    # ---------------------------------

    def train_all(self):

        outputs = {}

        for p in self.primitives:

            print("Training:", p)

            outputs[p] = self.train_one(p)

        return outputs
    
        # ---------------------------------
    # train one primitive using RF classifier with probability output
    # ---------------------------------

    def train_one_mc(self, primitive):

        # remove system:index
        roi_clean = self.roi.map(
            lambda f: f.select(
                f.propertyNames().remove("system:index")
            )
        )

        sample = self.image.sampleRegions(
            collection=roi_clean,
            properties=[primitive],
            scale=30,
            geometries=False
        )

        classifier = ee.Classifier.smileRandomForest(50)\
            .setOutputMode('PROBABILITY') #result will be in probability value for monte carlo testing

        trained = classifier.train(
            features=sample,
            classProperty=primitive,
            inputProperties=self.image.bandNames()
        )

        result = self.image.classify(trained)

        return result.rename(primitive)

    # ---------------------------------
    # train all primitives
    # ---------------------------------

    def train_all_mc(self):

        outputs = {}

        for p in self.primitives:

            print("Training:", p)

            outputs[p] = self.train_one_mc(p)

        return outputs

## Deterministic primitive layers

In [ ]:
# Generate primitive layers

# predictor image (from your previous step)
image = stacked_landsat

# modular training data
roi = training_fc

trainer = PrimitiveLayerTrainer(
    image=image,
    roi=roi
)

primitive_layers = trainer.train_all()

print(primitive_layers.keys())

## Probabilistic primitive layers

In [ ]:
# Generate primitive layers

trainer = PrimitiveLayerTrainer(
    image=image,
    roi=roi
)

primitive_layers_mc = trainer.train_all_mc()

print(primitive_layers_mc.keys())

In [ ]:
# 1. Stack dict → ee.Image
primitive_stack_mc = ee.Image.cat(list(primitive_layers_mc.values()))
print("Bands in stack:", primitive_stack_mc.bandNames().getInfo())

# 2. Centroid needs a maxError argument when geometry comes from a shapefile
test_point = aoi.geometry().centroid(maxError=1)

# 3. Sample one pixel to verify probability output
sample = primitive_stack_mc.sample(
    region=test_point,
    scale=30,
    numPixels=1
).first().toDictionary().getInfo()

print("Sample pixel values:")
for band, val in sample.items():
    status = "OK" if 0 < val < 1 else "BINARY — not probability!"
    print(f"  {band}: {val:.4f}  {status}")

In [ ]:
import geemap

m = geemap.Map()

m.centerObject(aoi, 12)

m.addLayer(aoi, {}, "AOI")

m.addLayer(
    training_fc,
    {"color": "red"},
    "Training points"
)

m.addLayer(
    primitive_layers["tree_pres"],
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "tree_pres"
)
m.addLayer(
    primitive_layers["buil_pres"],
    {"min": 0, "max": 1, "palette": ["white", "red"]},
    "buil_pres"
)
m.addLayer(
    primitive_layers["water_pres"],
    {"min": 0, "max": 1, "palette": ["white", "brown"]},
    "water_pres"
)

m

In [ ]:
# Visualize probabilistic primitive layers

m = geemap.Map()

m.centerObject(aoi, 12)

m.addLayer(aoi, {}, "AOI")


m.addLayer(
    training_fc,
    {"color": "red"},
    "Training points"
)

m.addLayer(
    primitive_layers_mc["tree_pres"],
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "tree_pres"
)
m.addLayer(
    primitive_layers_mc["buil_pres"],
    {"min": 0, "max": 1, "palette": ["white", "red"]},
    "buil_pres"
)
m.addLayer(
    primitive_layers_mc["water_pres"],
    {"min": 0, "max": 1, "palette": ["white", "brown"]},
    "water_pres"
)

m

In [ ]:
# Clean each primitive layer manually, removing all properties including system:index
primitive_layers_clean = {}
for k, v in primitive_layers.items():
    img = ee.Image(v).toFloat().rename(k).copyProperties(v, [])  # copy no properties
    primitive_layers_clean[k] = img

# Concatenate into a single image
primitive_image = ee.Image.cat(list(primitive_layers_clean.values()))

# List layer names for reference
# Original keys
layer_names = list(primitive_layers.keys())

# Manually remove "system:index" if it exists
if "system:index" in layer_names:
    layer_names.remove("system:index")

print(layer_names)

# Step 5: Define classification scheme and the class definition

1.  Each of the class definition inside the scheme is defined based on LCML/LUML
2.  The class definition is turned into a ruleset

In [ ]:
import pandas as pd

def load_scheme(csv_path, primitive_layers):
    """
    Load a single classification scheme from a CSV.
    Convert 1/0 in primitive columns to EE-compatible rule expressions,
    using the 'rule' column to specify combination type (none/and/or).

    Args:
        csv_path (str): Path to CSV.
        primitive_layers (list of str): Names of primitive layers to check in CSV.

    Returns:
        pd.DataFrame with columns: class_id, class_name, rule
    """
    import pandas as pd
    df = pd.read_csv(csv_path)

    required_cols = ["class_id", "class_name", "rule"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in {csv_path}")

    # Primitive columns are all except class_id, class_name, rule
    primitive_cols = [c for c in df.columns if c not in ["class_id", "class_name", "rule"]]

    rules = []
    for _, row in df.iterrows():
        conds = []

        for col in primitive_cols:
            if col not in primitive_layers:
                raise ValueError(f"Column '{col}' in CSV not found in primitive layers")
            val = row[col]
            if val == 1:
                conds.append(f"{col} == 1")
            elif val == 0:
                conds.append(f"{col} == 0")
            else:
                raise ValueError(f"Invalid value '{val}' in column '{col}', expected 0 or 1")

        combination_type = str(row["rule"]).lower()
        if combination_type == "and":
            rule_expr = " AND ".join(conds)
        elif combination_type == "or":
            rule_expr = " OR ".join(conds)
        elif combination_type == "none":
            # Take first non-trivial condition only
            rule_expr = conds[0] if conds else ""
        else:
            raise ValueError(f"Invalid rule '{row['rule']}' in row {row['class_id']}")

        rules.append(rule_expr)

    df["rule"] = rules
    return df[["class_id", "class_name", "rule"]]

In [ ]:
scheme1_rules = load_scheme("../data/test_data/scheme1.csv", layer_names)
scheme2_rules = load_scheme("../data/test_data/scheme2.csv", layer_names)

# print(scheme1_rules)
print(scheme2_rules)

In [ ]:
import ee
import io
import zipfile
import rasterio
import numpy as np
import pandas as pd
import requests
import warnings
from typing import Optional


class RuleSetClassifier:
    """
    Classify an EE primitive image using CSV-defined rules.
    Supports multiple schemes and rule-based methods.
    """

    def __init__(self, primitive_image: ee.Image, rules_df: pd.DataFrame, aoi):
        """
        Args:
            primitive_image (ee.Image): Stack of primitive layers.
            rules_df (pd.DataFrame): Rules table with columns:
                                     class_id, class_name, rule, scheme
            aoi (ee.FeatureCollection or ee.Geometry): Area of interest for clipping
        """
        self.primitive_image = primitive_image
        self.df = rules_df
        self.aoi = aoi

    # -------------------------------------------------------------------------
    # Deterministic classification
    # -------------------------------------------------------------------------

    def classify_scheme_deterministic(self, scheme_name: str) -> ee.Image:
        subset = self.df[self.df["scheme"] == scheme_name]
        if subset.empty:
            raise ValueError(f"No rules found for scheme '{scheme_name}'")

        aoi_geom = self.aoi.geometry() if hasattr(self.aoi, "geometry") else self.aoi
        result = ee.Image(0).rename("class_id").toFloat()

        band_names = self.primitive_image.bandNames().getInfo()
        band_dict = {b: self.primitive_image.select(b) for b in band_names}

        for _, row in subset.iterrows():
            class_id = float(row["class_id"])
            rule_expr = row["rule"].replace("AND", "&&").replace("OR", "||")
            try:
                mask = ee.Image().expression(rule_expr, band_dict).eq(1)
                result = result.where(mask, class_id)
            except Exception as e:
                print(f"Skipping class_id {class_id} due to EE expression error: {e}")

        return result.clip(aoi_geom)

    # -------------------------------------------------------------------------

    def classify_all_schemes(self) -> dict:
        """
        Run deterministic classification for all unique schemes in rules_df.
        Returns a dictionary of {scheme_name: ee.Image}.
        """
        results = {}
        for s in self.df["scheme"].unique():
            print(f"Classifying scheme: {s}")
            results[s] = self.classify_scheme_deterministic(s)
        return results

    # -------------------------------------------------------------------------
    # Monte Carlo classification (probabilistic primitives)
    # -------------------------------------------------------------------------

    @staticmethod
    def _evaluate_rule_numpy(
        rule_expr: str,
        band_arrays: dict,
    ) -> np.ndarray:
        """
        Evaluate a GEE-style boolean expression over named numpy arrays.
        Supports: &&  ||  AND  OR  ==  !=  >=  <=  >  <  numeric literals.
        Returns a boolean 2-D array.
        """
        expr = (
            rule_expr
            .replace("&&", " & ")
            .replace("||", " | ")
            .replace("AND", " & ")
            .replace("OR", " | ")
        )
        local_vars = {name: arr.astype(float) for name, arr in band_arrays.items()}
        try:
            result = eval(expr, {"__builtins__": {}}, local_vars)  # noqa: S307
            return np.asarray(result, dtype=bool)
        except Exception as exc:
            raise ValueError(f"Cannot evaluate rule '{rule_expr}': {exc}") from exc

    # -------------------------------------------------------------------------

    def _get_band_arrays(self, scale: int = 30) -> dict:
        """
        Pull all primitive bands from the EE image as numpy arrays using
        getDownloadURL. Works correctly with shapefile-derived AOIs loaded
        via geemap.shp_to_ee() (ee.FeatureCollection or ee.Geometry).
 
        The image is exported as a multi-band GeoTIFF zip, then read back
        with rasterio. Band order in the file matches bandNames() order.
 
        Returns dict[band_name -> (H, W) float32 array].
        """
        # Resolve geometry — works for both ee.FeatureCollection and ee.Geometry
        aoi_geom = self.aoi.geometry() if hasattr(self.aoi, "geometry") else self.aoi
        band_names = self.primitive_image.bandNames().getInfo()
 
        print(f"  Fetching {len(band_names)} bands via getDownloadURL "
              f"(scale={scale}m)...")
 
        url = self.primitive_image.getDownloadURL({
            "bands":  band_names,
            "region": aoi_geom,
            "scale":  scale,
            "format": "GEO_TIFF",
            "crs":    "EPSG:4326",
        })
 
        response = requests.get(url, stream=True, timeout=300)
        response.raise_for_status()
 
        raw = response.content
 
        band_arrays = {}
 
        # GEE returns either a raw GeoTIFF (single band) or a ZIP of per-band
        # GeoTIFFs depending on the number of bands requested.
        if raw[:4] == b"PK\x03\x04":
            # --- ZIP of individual single-band GeoTIFFs ----------------------
            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                tif_names = sorted(n for n in zf.namelist() if n.endswith(".tif"))
                if len(tif_names) != len(band_names):
                    raise ValueError(
                        f"Expected {len(band_names)} GeoTIFFs in ZIP, "
                        f"got {len(tif_names)}: {tif_names}"
                    )
                for band_name, tif_name in zip(band_names, tif_names):
                    with zf.open(tif_name) as f:
                        with rasterio.open(io.BytesIO(f.read())) as src:
                            band_arrays[band_name] = src.read(1).astype(np.float32)
        else:
            # --- Single multi-band GeoTIFF -----------------------------------
            with rasterio.open(io.BytesIO(raw)) as src:
                if src.count != len(band_names):
                    raise ValueError(
                        f"Expected {len(band_names)} bands in GeoTIFF, "
                        f"got {src.count}."
                    )
                for i, band_name in enumerate(band_names, start=1):
                    band_arrays[band_name] = src.read(i).astype(np.float32)
 
        shapes = {n: a.shape for n, a in band_arrays.items()}
        unique_shapes = set(shapes.values())
        if len(unique_shapes) > 1:
            raise ValueError(f"Band arrays have inconsistent shapes: {shapes}")
 
        h, w = next(iter(shapes.values()))
        print(f"  Downloaded arrays: {h} x {w} px  "
              f"({h * w:,} pixels per band)")
 
        return band_arrays

    # -------------------------------------------------------------------------

    def classify_scheme_monte_carlo(
        self,
        scheme_name: str,
        n_iterations: int = 200,
        nodata_value: float = 0.0,
        seed: Optional[int] = 42,
        scale: int = 30,
    ) -> dict:
        """
        Monte Carlo LULC classification from probabilistic primitive layers.

        Each primitive band contains per-pixel probabilities in [0, 1] as
        produced by GEE's smileRandomForest with .setOutputMode('PROBABILITY').
        In each iteration, every pixel is binarised by sampling Bernoulli(p),
        and the resulting binary primitives are passed through the deterministic
        ruleset — propagating per-pixel uncertainty into the final class map.

        Parameters
        ----------
        scheme_name : str
            Which classification scheme to use (must exist in self.df).
        n_iterations : int
            Number of Monte Carlo draws (200–500 is usually sufficient).
        nodata_value : float
            Class ID written to pixels that match no rule in a given iteration.
        seed : int or None
            Random seed for reproducibility.
        scale : int
            Pixel scale in metres used when pulling arrays from GEE.

        Returns
        -------
        dict with keys:
            'mode_map'    – (H, W) int array  : most-frequent class_id per pixel.
            'entropy_map' – (H, W) float array: Shannon entropy (nats); high
                            values indicate low confidence / high disagreement.
            'class_probs' – dict[class_id -> (H, W) float array]: fraction of
                            iterations each class was assigned per pixel.
            'n_iterations'– int: number of iterations run.
        """
        subset = self.df[self.df["scheme"] == scheme_name].copy()
        if subset.empty:
            raise ValueError(f"No rules found for scheme '{scheme_name}'")

        subset = subset.sort_values("class_id").reset_index(drop=True)

        # --- Pull probabilistic primitive arrays from GEE (once) --------------
        print(f"Pulling primitive arrays from GEE for scheme '{scheme_name}'...")
        band_arrays = self._get_band_arrays(scale=scale)

        # Validate that all bands are probability values in [0, 1]
        for band_name, arr in band_arrays.items():
            if arr.min() < 0 or arr.max() > 1:
                raise ValueError(
                    f"Band '{band_name}' contains values outside [0, 1]. "
                    "Ensure primitives are trained with .setOutputMode('PROBABILITY')."
                )

        sample_band = next(iter(band_arrays.values()))
        H, W = sample_band.shape

        all_class_ids = sorted(subset["class_id"].unique().tolist())
        if nodata_value not in all_class_ids:
            all_class_ids = [nodata_value] + all_class_ids

        class_id_to_idx = {cid: i for i, cid in enumerate(all_class_ids)}
        n_classes = len(all_class_ids)
        counts = np.zeros((n_classes, H, W), dtype=np.int32)

        rng = np.random.default_rng(seed)

        print(f"Running {n_iterations} Monte Carlo iterations...")
        for i in range(n_iterations):

            # --- 1. Sample binary realisations from each probabilistic primitive
            #
            #   p = 0.94  ->  almost always 1   (confident tree pixel)
            #   p = 0.53  ->  nearly coin flip  (uncertain pixel)
            #   p = 0.07  ->  almost always 0   (confident non-tree pixel)
            #
            binary_bands = {
                band_name: (rng.random((H, W)) < prob_arr).astype(np.uint8)
                for band_name, prob_arr in band_arrays.items()
            }

            # --- 2. Apply deterministic ruleset to sampled binary primitives ---
            iteration_result = np.full((H, W), nodata_value, dtype=float)

            for _, row in subset.iterrows():
                class_id = float(row["class_id"])
                try:
                    mask = self._evaluate_rule_numpy(row["rule"], binary_bands)
                    iteration_result[mask] = class_id
                except ValueError as exc:
                    warnings.warn(str(exc), stacklevel=2)

            # --- 3. Accumulate per-class pixel counts --------------------------
            for cid, idx in class_id_to_idx.items():
                counts[idx] += (iteration_result == cid).astype(np.int32)

        # -----------------------------------------------------------------------
        # Aggregate across iterations
        # -----------------------------------------------------------------------

        # Mode map: class with the highest iteration count per pixel
        best_idx = np.argmax(counts, axis=0)
        idx_to_class_id = np.array(all_class_ids, dtype=float)
        mode_map = idx_to_class_id[best_idx].astype(int)

        # Empirical class probabilities  p_c = count_c / n_iterations
        probs = counts.astype(float) / n_iterations  # (n_classes, H, W)

        # Shannon entropy  H = -sum(p * ln(p)),  convention: 0 * ln(0) = 0
        with np.errstate(divide="ignore", invalid="ignore"):
            log_probs = np.where(probs > 0, np.log(probs), 0.0)
        entropy_map = -np.sum(probs * log_probs, axis=0)  # (H, W)

        class_probs = {cid: probs[idx] for cid, idx in class_id_to_idx.items()}

        return {
            "mode_map": mode_map,
            "entropy_map": entropy_map,
            "class_probs": class_probs,
            "n_iterations": n_iterations,
        }

# Step 6: Generate LULC map

## Generate using deterministic ruleset

In [ ]:
scheme1_rules["scheme"] = "scheme1"  # add scheme column
scheme2_rules["scheme"] = "scheme2"  # add scheme column

# Initialize classifier
classifier1 = RuleSetClassifier(
    primitive_image=primitive_image,
    rules_df=scheme1_rules,
    aoi=aoi
)
classifier2 = RuleSetClassifier(
    primitive_image=primitive_image,
    rules_df=scheme2_rules,
    aoi=aoi
)


In [ ]:

# Deterministic classification
map1_det = classifier1.classify_scheme_deterministic("scheme1")
map2_det = classifier2.classify_scheme_deterministic("scheme2")

In [ ]:
"""
Validation for deterministic LULC classification.
Pulls the ee.Image result as a numpy array and produces
the same console summary + plots as validate_monte_carlo(),
so both approaches can be compared side by side.
"""

import io
import zipfile
import requests
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec


# ---------------------------------------------------------------------------
# Helper: pull a single-band ee.Image to numpy
# ---------------------------------------------------------------------------

def _ee_image_to_numpy(ee_image, aoi, scale: int = 30) -> np.ndarray:
    """
    Download a single-band ee.Image clipped to aoi as a numpy array.
    Works with ee.FeatureCollection or ee.Geometry AOIs (e.g. from
    geemap.shp_to_ee).
    """
    aoi_geom = aoi.geometry() if hasattr(aoi, "geometry") else aoi

    url = ee_image.getDownloadURL({
        "bands":  ee_image.bandNames().getInfo(),
        "region": aoi_geom,
        "scale":  scale,
        "format": "GEO_TIFF",
        "crs":    "EPSG:4326",
    })

    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    raw = response.content

    if raw[:4] == b"PK\x03\x04":
        with zipfile.ZipFile(io.BytesIO(raw)) as zf:
            tif_name = sorted(n for n in zf.namelist() if n.endswith(".tif"))[0]
            with zf.open(tif_name) as f:
                with rasterio.open(io.BytesIO(f.read())) as src:
                    return src.read(1).astype(np.float32)
    else:
        with rasterio.open(io.BytesIO(raw)) as src:
            return src.read(1).astype(np.float32)


# ---------------------------------------------------------------------------
# Main validation function
# ---------------------------------------------------------------------------

def validate_deterministic(
    det_ee_image,
    rules_df: pd.DataFrame,
    scheme_name: str,
    aoi,
    scale: int = 30,
    scheme_label: str = "",
):
    """
    Validation for a deterministic classification result.
    Produces the same console summary and plots as validate_monte_carlo()
    so the two approaches can be directly compared.

    Parameters
    ----------
    det_ee_image : ee.Image
        Output of classify_scheme_deterministic() — single band 'class_id'.
    rules_df : pd.DataFrame
        Rules table used for classification (to get class names).
    scheme_name : str
        Scheme key in rules_df.
    aoi : ee.FeatureCollection or ee.Geometry
        Area of interest — same object passed to RuleSetClassifier.
    scale : int
        Pixel scale in metres for downloading the result.
    scheme_label : str
        Display label for plot titles.

    Returns
    -------
    class_map : np.ndarray
        (H, W) int array of class IDs, matching mode_map shape from MC.
    """
    label  = scheme_label or scheme_name
    subset = rules_df[rules_df["scheme"] == scheme_name].copy()
    id_to_name  = dict(zip(subset["class_id"], subset["class_name"]))
    class_ids   = sorted(subset["class_id"].unique().tolist())

    # --- Pull ee.Image to numpy ---------------------------------------------
    print(f"Downloading deterministic result for '{label}'...")
    class_map = _ee_image_to_numpy(det_ee_image, aoi, scale=scale).astype(int)
    H, W = class_map.shape
    total_pixels = class_map.size

    # --- Console summary ----------------------------------------------------
    print(f"\n{'='*55}")
    print(f"  Validation — {label}  (deterministic)")
    print(f"{'='*55}")
    print(f"  Spatial extent  : {H} x {W} px")
    print(f"  Entropy         : 0.0000 nats  (no uncertainty — deterministic)")
    print(f"\n  Per-class area share (class map):")

    area_shares = {}
    for cid in [0] + class_ids:
        area_pct = 100 * (class_map == cid).sum() / total_pixels
        area_shares[cid] = area_pct
        name = id_to_name.get(cid, f"class {cid}")
        print(f"    [{int(cid):>2}] {name:<20}  area={area_pct:5.1f}%")

    print(f"{'='*55}\n")

    # --- Figure: class map + per-class area bar chart -----------------------
    n_cols = 2
    fig = plt.figure(figsize=(12, 5), constrained_layout=True)
    fig.suptitle(f"Deterministic validation — {label}", fontsize=13, fontweight="500")
    gs = GridSpec(1, n_cols, figure=fig)

    # Panel 1: spatial class map
    base_palette = [
        "#888780",  # 0 nodata   — gray
        "#1D9E75",  # 1          — teal
        "#378ADD",  # 2          — blue
        "#D85A30",  # 3          — coral
        "#BA7517",  # 4          — amber
        "#7F77DD",  # 5          — purple
        "#639922",  # 6          — green
    ]
    all_ids  = [0] + class_ids
    cmap     = plt.cm.colors.ListedColormap(
        [base_palette[i % len(base_palette)] for i in range(len(all_ids))]
    )
    bounds   = [i - 0.5 for i in range(len(all_ids) + 1)]
    norm     = plt.cm.colors.BoundaryNorm(bounds, cmap.N)

    # Remap class IDs to contiguous indices for imshow
    display_map = np.zeros_like(class_map)
    for idx, cid in enumerate(all_ids):
        display_map[class_map == cid] = idx

    ax_map = fig.add_subplot(gs[0, 0])
    im = ax_map.imshow(display_map, cmap=cmap, norm=norm, interpolation="nearest")
    cbar = plt.colorbar(im, ax=ax_map, ticks=range(len(all_ids)),
                        fraction=0.046, pad=0.04)
    cbar.set_ticklabels(
        [f"[{int(cid)}] {id_to_name.get(cid, 'nodata')}" for cid in all_ids]
    )
    ax_map.set_title("Class map", fontsize=11)
    ax_map.axis("off")

    # Panel 2: area bar chart (only named classes, skip nodata=0)
    ax_bar = fig.add_subplot(gs[0, 1])
    named_ids   = [cid for cid in class_ids if area_shares.get(cid, 0) > 0]
    named_names = [id_to_name.get(cid, f"class {cid}") for cid in named_ids]
    named_areas = [area_shares[cid] for cid in named_ids]
    bar_colors  = [base_palette[(i + 1) % len(base_palette)]
                   for i, cid in enumerate(class_ids)
                   if area_shares.get(cid, 0) > 0]

    bars = ax_bar.barh(named_names, named_areas, color=bar_colors,
                       edgecolor="none", height=0.5)
    ax_bar.set_xlabel("Area share (%)", fontsize=11)
    ax_bar.set_title("Per-class area share", fontsize=11)
    ax_bar.set_xlim(0, 100)

    for bar, pct in zip(bars, named_areas):
        ax_bar.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f"{pct:.1f}%", va="center", fontsize=9)

    plt.show()

    return class_map


In [ ]:

# Step 1: validate deterministic results and pull as numpy
det_map1 = validate_deterministic(
    det_ee_image = map1_det,
    rules_df     = scheme1_rules,
    scheme_name  = "scheme1",
    aoi          = aoi,
    scale        = 30,
    scheme_label = "Scheme 1",
)

det_map2 = validate_deterministic(
    det_ee_image = map2_det,
    rules_df     = scheme2_rules,
    scheme_name  = "scheme2",
    aoi          = aoi,
    scale        = 30,
    scheme_label = "Scheme 2",
)

## Generate using Monte Carlo simulation

In [ ]:
classifier1_mc = RuleSetClassifier(
    primitive_image=primitive_stack_mc,   # stacked ee.Image, not the dict
    rules_df=scheme1_rules,
    aoi=aoi
)
classifier2_mc = RuleSetClassifier(
    primitive_image=primitive_stack_mc,
    rules_df=scheme2_rules,
    aoi=aoi
)

In [ ]:
results1 = classifier1_mc.classify_scheme_monte_carlo(
    scheme_name="scheme1",
    n_iterations=300,
    seed=42,
    scale=30,
)
 
results2 = classifier2_mc.classify_scheme_monte_carlo(
    scheme_name="scheme2",
    n_iterations=300,
    seed=42,
    scale=30,
)

In [ ]:
# Unpack MC outputs
map1_mode    = results1["mode_map"]
map1_entropy = results1["entropy_map"]
map1_probs   = results1["class_probs"]
 
map2_mode    = results2["mode_map"]
map2_entropy = results2["entropy_map"]
map2_probs   = results2["class_probs"]
 

In [ ]:
"""
Implementation: Monte Carlo LULC classification
with validation checks and geemap visualization.
"""
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

# ===========================================================================
# VALIDATION
# ===========================================================================

def validate_monte_carlo(
    results: dict,
    rules_df: pd.DataFrame,
    scheme_name: str,
    entropy_threshold: float = 0.5,
    scheme_label: str = "",
):
    """
    Sanity checks for a Monte Carlo result:
      1. Entropy map — distribution + high-uncertainty pixel fraction
      2. Per-class probability histograms

    Parameters
    ----------
    results : dict
        Output of classify_scheme_monte_carlo().
    rules_df : pd.DataFrame
        Rules table used for classification (to get class names).
    scheme_name : str
        Scheme key in rules_df.
    entropy_threshold : float
        Pixels above this entropy (nats) are considered high-uncertainty.
        For a binary split the maximum entropy is ln(2) ≈ 0.693 nats.
        A good default is 0.5 (≈72% of maximum for a 2-class case).
    scheme_label : str
        Display label for plot titles.
    """
    mode_map    = results["mode_map"]
    entropy_map = results["entropy_map"]
    class_probs = results["class_probs"]
    n_iter      = results["n_iterations"]

    subset = rules_df[rules_df["scheme"] == scheme_name].copy()
    id_to_name = dict(zip(subset["class_id"], subset["class_name"]))

    label = scheme_label or scheme_name
    n_classes = len(class_probs)

    # -----------------------------------------------------------------------
    # Console summary
    # -----------------------------------------------------------------------
    total_pixels = mode_map.size
    high_unc_pixels = (entropy_map > entropy_threshold).sum()
    high_unc_pct = 100 * high_unc_pixels / total_pixels

    print(f"\n{'='*55}")
    print(f"  Validation — {label}  ({n_iter} iterations)")
    print(f"{'='*55}")
    print(f"  Spatial extent  : {mode_map.shape[0]} x {mode_map.shape[1]} px")
    print(f"  Entropy range   : {entropy_map.min():.4f} – {entropy_map.max():.4f} nats")
    print(f"  Mean entropy    : {entropy_map.mean():.4f} nats")
    print(f"  High-uncertainty pixels (entropy > {entropy_threshold}): "
          f"{high_unc_pixels:,}  ({high_unc_pct:.1f}%)")
    print(f"\n  Per-class area share (mode map):")
    for cid, prob_map in class_probs.items():
        area_pct = 100 * (mode_map == cid).sum() / total_pixels
        mean_conf = prob_map.mean()
        name = id_to_name.get(cid, f"class {cid}")
        print(f"    [{int(cid):>2}] {name:<20}  "
              f"area={area_pct:5.1f}%   mean prob={mean_conf:.3f}")
    print(f"{'='*55}\n")

    # -----------------------------------------------------------------------
    # Figure layout:  entropy row + per-class prob histograms
    # -----------------------------------------------------------------------
    n_cols   = min(n_classes, 4)
    n_rows   = 2 + (n_classes - 1) // n_cols   # row 0: entropy; rows 1+: hists
    fig = plt.figure(figsize=(5 * n_cols, 4 * n_rows), constrained_layout=True)
    fig.suptitle(f"Monte Carlo validation — {label}", fontsize=13, fontweight="500")
    gs = GridSpec(n_rows, n_cols, figure=fig)

    # --- Row 0: entropy distribution + spatial map --------------------------
    ax_hist = fig.add_subplot(gs[0, :n_cols // 2])
    ax_hist.hist(
        entropy_map.ravel(), bins=60, color="#5DCAA5", edgecolor="none", alpha=0.85
    )
    ax_hist.axvline(entropy_threshold, color="#D85A30", linewidth=1.5,
                    linestyle="--", label=f"threshold = {entropy_threshold}")
    ax_hist.axvline(entropy_map.mean(), color="#7F77DD", linewidth=1.5,
                    linestyle="-", label=f"mean = {entropy_map.mean():.3f}")
    ax_hist.set_xlabel("Shannon entropy (nats)", fontsize=11)
    ax_hist.set_ylabel("Pixel count", fontsize=11)
    ax_hist.set_title("Entropy distribution", fontsize=11)
    ax_hist.legend(fontsize=9)

    ax_map = fig.add_subplot(gs[0, n_cols // 2:])
    emap = ax_map.imshow(entropy_map, cmap="YlOrRd", vmin=0, vmax=entropy_map.max())
    plt.colorbar(emap, ax=ax_map, fraction=0.046, pad=0.04, label="entropy (nats)")
    # Overlay high-uncertainty mask
    high_unc_mask = np.where(entropy_map > entropy_threshold, 1.0, np.nan)
    ax_map.imshow(high_unc_mask, cmap="cool", alpha=0.4, vmin=0, vmax=1)
    ax_map.set_title(
        f"Entropy map  (cyan = high uncertainty, {high_unc_pct:.1f}% of pixels)",
        fontsize=11
    )
    ax_map.axis("off")

    # --- Rows 1+: per-class probability histograms --------------------------
    class_items = [(cid, prob_map) for cid, prob_map in class_probs.items()
                   if cid != 0.0]   # skip nodata

    for i, (cid, prob_map) in enumerate(class_items):
        row = 1 + i // n_cols
        col = i % n_cols
        ax = fig.add_subplot(gs[row, col])
        name = id_to_name.get(cid, f"class {cid}")

        ax.hist(prob_map.ravel(), bins=50, color="#378ADD", edgecolor="none", alpha=0.85)
        ax.axvline(0.5, color="#E24B4A", linewidth=1.2, linestyle="--", label="p = 0.5")
        ax.axvline(prob_map.mean(), color="#BA7517", linewidth=1.2,
                   label=f"mean = {prob_map.mean():.3f}")
        ax.set_title(f"[{int(cid)}] {name}", fontsize=10)
        ax.set_xlabel("P(class assigned)", fontsize=9)
        ax.set_ylabel("Pixel count", fontsize=9)
        ax.legend(fontsize=8)

        # Annotate bimodality if present: most mass near 0 and near 1 is healthy
        near_zero = (prob_map < 0.2).mean()
        near_one  = (prob_map > 0.8).mean()
        mid       = ((prob_map >= 0.2) & (prob_map <= 0.8)).mean()
        verdict = (
            "bimodal (healthy)" if (near_zero + near_one > 0.7)
            else "flat/uncertain" if mid > 0.5
            else "skewed"
        )
        ax.text(0.97, 0.95, verdict, transform=ax.transAxes,
                ha="right", va="top", fontsize=8,
                color="#0F6E56" if "healthy" in verdict else "#993C1D")

    #plt.savefig(f"validation_{scheme_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    #print(f"  Saved: validation_{scheme_name}.png")


# Run validation for both schemes
validate_monte_carlo(results1, scheme1_rules, "scheme1",
                     entropy_threshold=0.5, scheme_label="Scheme 1")
validate_monte_carlo(results2, scheme2_rules, "scheme2",
                     entropy_threshold=0.5, scheme_label="Scheme 2")



## Compare deterministic vs mc

In [ ]:

# ---------------------------------------------------------------------------
# Side-by-side comparison helper
# ---------------------------------------------------------------------------

def compare_det_vs_mc(
    det_class_map: np.ndarray,
    mc_results: dict,
    rules_df: pd.DataFrame,
    scheme_name: str,
    scheme_label: str = "",
):
    """
    Print a side-by-side area share comparison and compute pixel-level
    agreement between the deterministic map and the MC mode map.

    Parameters
    ----------
    det_class_map : np.ndarray
        Output of validate_deterministic() — (H, W) int array.
    mc_results : dict
        Output of classify_scheme_monte_carlo().
    rules_df : pd.DataFrame
        Rules table (for class names).
    scheme_name : str
        Scheme key in rules_df.
    scheme_label : str
        Display label.
    """
    label      = scheme_label or scheme_name
    subset     = rules_df[rules_df["scheme"] == scheme_name].copy()
    id_to_name = dict(zip(subset["class_id"], subset["class_name"]))
    class_ids  = sorted(subset["class_id"].unique().tolist())

    mode_map    = mc_results["mode_map"]
    entropy_map = mc_results["entropy_map"]
    total       = det_class_map.size

    # Overall pixel agreement
    agreement     = (det_class_map == mode_map).sum()
    agreement_pct = 100 * agreement / total

    print(f"\n{'='*55}")
    print(f"  Deterministic vs MC — {label}")
    print(f"{'='*55}")
    print(f"  Overall pixel agreement : {agreement:,} / {total:,}  "
          f"({agreement_pct:.1f}%)")
    print(f"  Mean MC entropy         : {entropy_map.mean():.4f} nats")
    print(f"\n  {'Class':<22}  {'Det area':>9}  {'MC area':>9}  {'Δ':>7}")
    print(f"  {'-'*52}")

    for cid in class_ids:
        name     = id_to_name.get(cid, f"class {cid}")
        det_pct  = 100 * (det_class_map == cid).sum() / total
        mc_pct   = 100 * (mode_map == cid).sum() / total
        delta    = mc_pct - det_pct
        sign     = "+" if delta >= 0 else ""
        print(f"  [{int(cid):>2}] {name:<18}  "
              f"{det_pct:>8.1f}%  {mc_pct:>8.1f}%  {sign}{delta:>5.1f}%")

    print(f"{'='*55}\n")

    # --- Disagreement map ---------------------------------------------------
    disagree_mask = (det_class_map != mode_map).astype(float)
    disagree_pct  = 100 * disagree_mask.mean()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    fig.suptitle(f"Deterministic vs MC comparison — {label}",
                 fontsize=13, fontweight="500")

    # Deterministic class map
    axes[0].imshow(det_class_map, cmap="tab10", interpolation="nearest")
    axes[0].set_title("Deterministic", fontsize=11)
    axes[0].axis("off")

    # MC mode map
    axes[1].imshow(mode_map, cmap="tab10", interpolation="nearest")
    axes[1].set_title("MC mode map", fontsize=11)
    axes[1].axis("off")

    # Disagreement map — white=agree, red=disagree
    # Overlay MC entropy as intensity so high-entropy disagreements stand out
    disagree_display = np.where(disagree_mask == 1, entropy_map, 0)
    im = axes[2].imshow(disagree_display, cmap="YlOrRd",
                        vmin=0, vmax=entropy_map.max(),
                        interpolation="nearest")
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04,
                 label="MC entropy (nats)")
    axes[2].set_title(
        f"Disagreement pixels ({disagree_pct:.1f}%)\ncoloured by MC entropy",
        fontsize=11
    )
    axes[2].axis("off")

    plt.show()



In [ ]:

# Step 2: compare deterministic vs MC side by side
compare_det_vs_mc(
    det_class_map = det_map1,
    mc_results    = results1,
    rules_df      = scheme1_rules,
    scheme_name   = "scheme1",
    scheme_label  = "Scheme 1",
)

compare_det_vs_mc(
    det_class_map = det_map2,
    mc_results    = results2,
    rules_df      = scheme2_rules,
    scheme_name   = "scheme2",
    scheme_label  = "Scheme 2",
)